<!-- AUTO-GENERATED from notebooks/02_eda_phase0.ipynb — do not edit on disk. Run: python scripts/sync_kaggle_eda.py --push -->
## Kaggle bootstrap

Installs this repo on Kaggle. Skipped when running locally.

Repo: `https://github.com/sh0ch/RSNA-knee-abnormality-detection.git`

Sync from your machine: `python scripts/sync_kaggle_eda.py --push`


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/sh0ch/RSNA-knee-abnormality-detection.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working/rsna_knee_repo"

if os.environ.get("KAGGLE_KERNEL_RUN") == "True":
    if not os.path.exists(WORK_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, WORK_DIR],
            check=True,
        )
    subprocess.run(["pip", "install", "-q", "-e", f"{WORK_DIR}[dev]"], check=True)
    sys.path.insert(0, f"{WORK_DIR}/src")
else:
    print("Local run — skipping Kaggle bootstrap.")


# Phase 0 — Exploratory Data Analysis

RSNA 2026 Knee Abnormality Detection · macro ROC-AUC across 12 multilabel targets.

**Run on Kaggle** with the competition data attached for meaningful results. Locally, use synthetic sample data (`python scripts/create_sample_data.py`) to verify the notebook runs end-to-end.

This notebook answers the Phase 0 questions:
1. Label prevalence and class imbalance
2. Label co-occurrence patterns
3. Series structure per study (`FluidSensitiveSeries`)
4. Radiology report characteristics (train only)
5. Train vs. test distribution shifts
6. DICOM geometry (slice counts, spatial size) on a sample

## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Kaggle: repo cloned above; local: search from notebooks/
WORK_DIR = Path("/kaggle/working/rsna_knee_repo")
if WORK_DIR.is_dir():
    REPO_ROOT = WORK_DIR
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "src" / "rsna_knee").is_dir():
            REPO_ROOT = candidate
            break
    else:
        REPO_ROOT = Path.cwd().parent

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from rsna_knee.constants import (
    FLUID_COL,
    PATIENT_SEX_COL,
    REPORT_COL,
    SERIES_ID_COL,
    STUDY_ID_COL,
    TARGET_DISPLAY_NAMES,
    TARGET_LABELS,
)
from rsna_knee.data.dicom_io import load_series_volume, series_metadata_summary
from rsna_knee.utils.paths import (
    default_data_root,
    is_kaggle_kernel,
    series_dir,
    test_csv,
    test_series_csv,
    train_csv,
    train_series_csv,
)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

DATA_ROOT = default_data_root()
RNG = np.random.default_rng(42)

print(f"Environment : {'Kaggle kernel' if is_kaggle_kernel() else 'local'}")
print(f"Data root   : {DATA_ROOT}")

In [ ]:
train = pd.read_csv(train_csv(DATA_ROOT))
train_series = pd.read_csv(train_series_csv(DATA_ROOT))
test = pd.read_csv(test_csv(DATA_ROOT))
test_series = pd.read_csv(test_series_csv(DATA_ROOT))

print(f"Train studies : {len(train):,}")
print(f"Train series  : {len(train_series):,}")
print(f"Test studies  : {len(test):,}")
print(f"Test series   : {len(test_series):,}")

## 1. Dataset overview

In [ ]:
display(train.head(3))
display(train_series.head(3))

In [ ]:
overview = pd.DataFrame(
    {
        "train_studies": [len(train)],
        "train_series": [len(train_series)],
        "test_studies": [len(test)],
        "test_series": [len(test_series)],
        "series_per_study_train_mean": [train_series.groupby(STUDY_ID_COL).size().mean()],
        "series_per_study_test_mean": [test_series.groupby(STUDY_ID_COL).size().mean()],
        "unique_patients_train_sex": [train[PATIENT_SEX_COL].nunique() if PATIENT_SEX_COL in train else np.nan],
    }
)
overview

In [ ]:
missing = pd.DataFrame(
    {
        "train_missing": train.isna().sum(),
        "train_series_missing": train_series.isna().sum(),
        "test_missing": test.isna().sum(),
        "test_series_missing": test_series.isna().sum(),
    }
)
missing[missing.sum(axis=1) > 0]

## 2. Label prevalence

Macro ROC-AUC treats every label equally — rare findings still matter.

In [ ]:
label_stats = []
for label in TARGET_LABELS:
    positives = int(train[label].sum())
    total = len(train)
    label_stats.append(
        {
            "label": label,
            "display_name": TARGET_DISPLAY_NAMES[label],
            "positives": positives,
            "negatives": total - positives,
            "positive_rate": positives / total,
            "prevalence_pct": 100 * positives / total,
        }
    )

label_df = pd.DataFrame(label_stats).sort_values("positive_rate", ascending=False)
label_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(label_df["display_name"], label_df["prevalence_pct"], color="steelblue")
ax.set_xlabel("Positive rate (%)")
ax.set_title("Label prevalence (train)")
ax.invert_yaxis()

for bar, rate in zip(bars, label_df["prevalence_pct"], strict=True):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, f"{rate:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

labels_per_study = train[TARGET_LABELS].sum(axis=1)
print("Labels per study — mean: {:.2f}, median: {:.0f}, max: {:.0f}".format(
    labels_per_study.mean(), labels_per_study.median(), labels_per_study.max()
))
print("Studies with zero positive labels:", int((labels_per_study == 0).sum()))

## 3. Label co-occurrence

Joint positive rate: fraction of studies where both labels are positive.

In [ ]:
y = train[TARGET_LABELS].to_numpy(dtype=float)
n = len(train)
cooccur = (y.T @ y) / n

cooccur_df = pd.DataFrame(cooccur, index=TARGET_LABELS, columns=TARGET_LABELS)
cooccur_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cooccur_df.values, cmap="YlOrRd", vmin=0, vmax=cooccur_df.values.max())
ax.set_xticks(range(len(TARGET_LABELS)), TARGET_LABELS, rotation=90, fontsize=8)
ax.set_yticks(range(len(TARGET_LABELS)), TARGET_LABELS, fontsize=8)
ax.set_title("Joint positive rate (train)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="P(label_i ∧ label_j)")
plt.tight_layout()
plt.show()

In [ ]:
# Top co-occurring pairs (excluding diagonal)
pairs = []
for i, a in enumerate(TARGET_LABELS):
    for j, b in enumerate(TARGET_LABELS):
        if j <= i:
            continue
        pairs.append({"label_a": a, "label_b": b, "joint_rate": cooccur[i, j]})

top_pairs = pd.DataFrame(pairs).sort_values("joint_rate", ascending=False).head(10)
top_pairs

## 4. Series structure per study

Each study may contain multiple MRI sequences. `FluidSensitiveSeries == 1` marks PD/STIR-like sequences useful for soft-tissue findings.

In [ ]:
series_per_study = train_series.groupby(STUDY_ID_COL).size().rename("num_series")
fluid_per_study = (
    train_series.groupby(STUDY_ID_COL)[FLUID_COL].sum().rename("num_fluid_sensitive")
    if FLUID_COL in train_series.columns
    else pd.Series(dtype=float)
)

series_summary = pd.concat([series_per_study, fluid_per_study], axis=1)
series_summary.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

series_per_study.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_xlabel("Series per study")
axes[0].set_ylabel("Count")
axes[0].set_title("Train: series count distribution")

if FLUID_COL in train_series.columns:
    fluid_per_study.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="coral")
    axes[1].set_xlabel("Fluid-sensitive series per study")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Train: fluid-sensitive series per study")
    pct_with_fluid = 100 * (fluid_per_study > 0).mean()
    print(f"Studies with ≥1 fluid-sensitive series: {pct_with_fluid:.1f}%")
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
if FLUID_COL in train_series.columns:
    fluid_rate = train_series[FLUID_COL].mean()
    print(f"Series-level fluid-sensitive rate: {100 * fluid_rate:.1f}%")

    # Label prevalence conditional on having fluid-sensitive series
    studies_with_fluid = set(fluid_per_study[fluid_per_study > 0].index)
    mask = train[STUDY_ID_COL].isin(studies_with_fluid)
    cond_rates = train.loc[mask, TARGET_LABELS].mean().rename("rate_with_fluid")
    base_rates = train[TARGET_LABELS].mean().rename("rate_all")
    pd.concat([base_rates, cond_rates], axis=1)

## 5. Radiology reports (train only)

Reports are **not available at inference**. Use this section to gauge whether text is worth exploiting during training (distillation, auxiliary loss, etc.).

In [ ]:
if REPORT_COL in train.columns:
    reports = train[REPORT_COL].fillna("").astype(str)
    report_chars = reports.str.len()
    report_words = reports.str.split().str.len()

    report_stats = pd.Series(
        {
            "missing_or_empty": int((reports.str.strip() == "").sum()),
            "char_mean": report_chars.mean(),
            "char_median": report_chars.median(),
            "char_p95": report_chars.quantile(0.95),
            "word_mean": report_words.mean(),
            "word_median": report_words.median(),
        }
    )
    display(report_stats.to_frame("value"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    report_words.clip(upper=report_words.quantile(0.99)).hist(bins=40, ax=axes[0], color="steelblue")
    axes[0].set_xlabel("Words per report (clipped at p99)")
    axes[0].set_title("Report length distribution")

    labels_per_study.hist(bins=range(0, int(labels_per_study.max()) + 2), ax=axes[1], color="coral", align="left")
    axes[1].set_xlabel("Positive labels per study")
    axes[1].set_title("Multilabel density")
    plt.tight_layout()
    plt.show()

    print("Sample report:\n", reports.iloc[0][:500], "..." if len(reports.iloc[0]) > 500 else "")
else:
    print("No Report column in train.csv")

## 6. Train vs. test metadata

Check for distribution shifts that could affect generalization. Test labels are hidden during scoring.

In [ ]:
compare_rows = []

if PATIENT_SEX_COL in train.columns and PATIENT_SEX_COL in test.columns:
    for sex in sorted(set(train[PATIENT_SEX_COL].dropna()) | set(test[PATIENT_SEX_COL].dropna())):
        compare_rows.append(
            {
                "feature": PATIENT_SEX_COL,
                "value": sex,
                "train_pct": 100 * (train[PATIENT_SEX_COL] == sex).mean(),
                "test_pct": 100 * (test[PATIENT_SEX_COL] == sex).mean(),
            }
        )

train_sps = train_series.groupby(STUDY_ID_COL).size()
test_sps = test_series.groupby(STUDY_ID_COL).size()
compare_rows.append(
    {
        "feature": "series_per_study",
        "value": "mean",
        "train_pct": train_sps.mean(),
        "test_pct": test_sps.mean(),
    }
)
compare_rows.append(
    {
        "feature": "series_per_study",
        "value": "median",
        "train_pct": train_sps.median(),
        "test_pct": test_sps.median(),
    }
)

if FLUID_COL in train_series.columns and FLUID_COL in test_series.columns:
    compare_rows.append(
        {
            "feature": "fluid_sensitive_series_rate",
            "value": "series-level",
            "train_pct": 100 * train_series[FLUID_COL].mean(),
            "test_pct": 100 * test_series[FLUID_COL].mean(),
        }
    )

compare_df = pd.DataFrame(compare_rows)
compare_df["delta"] = compare_df["test_pct"] - compare_df["train_pct"]
compare_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
train_sps.plot(kind="hist", bins=range(1, int(max(train_sps.max(), test_sps.max())) + 2), ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title("Train: series per study")
axes[0].set_xlabel("Series count")

test_sps.plot(kind="hist", bins=range(1, int(max(train_sps.max(), test_sps.max())) + 2), ax=axes[1], color="coral", alpha=0.8)
axes[1].set_title("Test: series per study")
axes[1].set_xlabel("Series count")
plt.tight_layout()
plt.show()

## 7. DICOM geometry (sampled)

Reads a random subset of series to estimate slice counts and spatial dimensions. On Kaggle, increase `DICOM_SAMPLE_SIZE` for tighter estimates.

In [ ]:
DICOM_SAMPLE_SIZE = 200 if is_kaggle_kernel() else min(8, len(train_series))

sample_idx = RNG.choice(len(train_series), size=min(DICOM_SAMPLE_SIZE, len(train_series)), replace=False)
sample_series = train_series.iloc[sample_idx].reset_index(drop=True)

dicom_rows = []
errors = 0

for _, row in sample_series.iterrows():
    study_uid = row[STUDY_ID_COL]
    series_uid = row[SERIES_ID_COL]
    path = series_dir(DATA_ROOT, split="train") / study_uid / series_uid
    try:
        volume, datasets = load_series_volume(path)
        meta = series_metadata_summary(datasets)
        dicom_rows.append(
            {
                STUDY_ID_COL: study_uid,
                SERIES_ID_COL: series_uid,
                "num_slices": volume.shape[0],
                "height": volume.shape[1],
                "width": volume.shape[2],
                "slice_thickness": meta.get("slice_thickness"),
                FLUID_COL: row.get(FLUID_COL, np.nan),
            }
        )
    except Exception as exc:
        errors += 1
        if errors <= 3:
            print(f"Skip {path.name}: {exc}")

dicom_df = pd.DataFrame(dicom_rows)
print(f"Loaded {len(dicom_df)} / {len(sample_series)} sampled series ({errors} errors)")
dicom_df.describe()

In [ ]:
if not dicom_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    dicom_df["num_slices"].plot(kind="hist", bins=30, ax=axes[0], color="steelblue")
    axes[0].set_xlabel("Slices per series")
    axes[0].set_title("Depth distribution")

    dicom_df["height"].plot(kind="hist", bins=30, ax=axes[1], color="coral")
    axes[1].set_xlabel("Height (pixels)")
    axes[1].set_title("Height distribution")

    dicom_df["width"].plot(kind="hist", bins=30, ax=axes[2], color="seagreen")
    axes[2].set_xlabel("Width (pixels)")
    axes[2].set_title("Width distribution")

    plt.tight_layout()
    plt.show()

    if FLUID_COL in dicom_df.columns and dicom_df[FLUID_COL].notna().any():
        by_fluid = dicom_df.groupby(FLUID_COL)["num_slices"].agg(["mean", "median", "count"])
        print("Slices by fluid-sensitive flag:")
        display(by_fluid)

## 8. Data card — preprocessing decisions

Fill in observations after running on full Kaggle data. These become fixed choices for Phase 1.

In [ ]:
rarest = label_df.sort_values("positive_rate").iloc[0]
most_common = label_df.iloc[0]

data_card = {
    "environment": "Kaggle" if is_kaggle_kernel() else "local sample",
    "n_train_studies": len(train),
    "n_train_series": len(train_series),
    "mean_series_per_study": float(series_per_study.mean()),
    "pct_studies_with_fluid_series": float(100 * (fluid_per_study > 0).mean()) if len(fluid_per_study) else None,
    "most_common_label": most_common["label"],
    "most_common_rate_pct": float(100 * most_common["positive_rate"]),
    "rarest_label": rarest["label"],
    "rarest_rate_pct": float(100 * rarest["positive_rate"]),
    "mean_labels_per_study": float(labels_per_study.mean()),
    "dicom_sample_size": len(dicom_df),
    "median_slices_sampled": float(dicom_df["num_slices"].median()) if not dicom_df.empty else None,
    "median_height_sampled": float(dicom_df["height"].median()) if not dicom_df.empty else None,
    "median_width_sampled": float(dicom_df["width"].median()) if not dicom_df.empty else None,
}

print("=" * 60)
print("DATA CARD SUMMARY")
print("=" * 60)
for key, value in data_card.items():
    print(f"  {key:32s} {value}")

print("\nSuggested Phase 1 defaults (edit after full EDA):")
print("  • Aggregation  : one fluid-sensitive series per study (fallback: first series)")
print("  • Volume shape : center crop/pad to [32, 256, 256] (see configs/default.yaml)")
print("  • CV strategy  : 5-fold, stratified by multilabel presence or study ID hash")
print("  • Loss         : BCEWithLogits; consider pos_weight for rare labels")
print("  • Text branch  : defer until image-only baseline submits successfully")